In [4]:
# ==========================================================
# BRANCHNET (CNN-BASED BRANCH PREDICTOR WITH FEDERATED LEARNING)
# ==========================================================

import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# ==========================================================
# SETTINGS
# ==========================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DATA_DIR = "6_clients"
FL_ROUNDS = 50
BATCH_SIZE = 256
LR = 0.001
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ==========================================================
# CONFIG
# ==========================================================
class BranchNetConfig:
    def __init__(self, input_dim: int, num_slices: int = 3):
        self.history_sizes = [37, 77, 152][:num_slices]
        self.conv_channels = [4, 5, 5][:num_slices]
        self.pooling_widths = [7, 15, 30][:num_slices]
        self.embedding_dim = 32
        self.conv_width = 3
        self.hidden_neurons = 10
        self.input_dim = input_dim
        self.num_slices = num_slices

# ==========================================================
# DATA
# ==========================================================
def load_csv(path):
    df = pd.read_csv(path)
    X = df.iloc[:, :-1].values.astype(np.float32)
    y = df.iloc[:, -1].values.astype(np.int64)
    return X, y

def make_loader(X, y):
    ds = TensorDataset(torch.tensor(X), torch.tensor(y))
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True)

# ==========================================================
# MODEL
# ==========================================================
class BranchNetSlice(nn.Module):
    def __init__(self, config, idx):
        super().__init__()
        self.history_size = config.history_sizes[idx]
        self.conv_channels = config.conv_channels[idx]
        self.pooling_width = config.pooling_widths[idx]

        self.embedding = nn.Embedding(config.input_dim, config.embedding_dim)

        self.conv = nn.Conv1d(
            config.embedding_dim,
            self.conv_channels,
            kernel_size=config.conv_width,
            padding=1
        )

        self.bn = nn.BatchNorm1d(self.conv_channels)
        self.pool = nn.AvgPool1d(self.pooling_width)

    def forward(self, x):
        x = self.embedding(x.long()).transpose(1, 2)
        x = torch.relu(self.bn(self.conv(x)))
        x = self.pool(x)
        return x

class BranchNet(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.slices = nn.ModuleList([BranchNetSlice(config, i) for i in range(config.num_slices)])

        # 🔥 CORRECT: infer feature size using dummy forward
        with torch.no_grad():
            dummy = torch.zeros(1, config.input_dim)
            outs = []
            idx = 0
            for s in self.slices:
                h = s.history_size
                xi = dummy[:, idx:idx+h]
                idx += h
                out = s(xi.long())
                outs.append(out.view(1, -1))
            total_features = torch.cat(outs, dim=1).shape[1]

        self.fc1 = nn.Linear(total_features, config.hidden_neurons)
        self.fc2 = nn.Linear(config.hidden_neurons, 1)

    def forward(self, x):
        outs = []
        idx = 0

        for s in self.slices:
            h = s.history_size
            xi = x[:, idx:idx+h]
            idx += h

            out = s(xi)
            outs.append(out.view(x.size(0), -1))

        x = torch.cat(outs, dim=1)
        x = torch.relu(self.fc1(x))
        return self.fc2(x).squeeze(1)

# ==========================================================
# EVAL
# ==========================================================
def eval_model(model, loader):
    model.eval()
    preds = []

    with torch.no_grad():
        for Xb, _ in loader:
            Xb = Xb.to(DEVICE)
            p = torch.sigmoid(model(Xb)) >= 0.5
            preds.extend(p.cpu().numpy())

    return np.array(preds)

def accuracy(preds, labels):
    return (preds == labels).mean() * 100

# ==========================================================
# TRAIN
# ==========================================================
def train_client(model, loader):
    optimizer = optim.Adam(model.parameters(), lr=LR)
    loss_fn = nn.BCEWithLogitsLoss()

    model.train()
    total_loss = 0

    for Xb, yb in loader:
        Xb, yb = Xb.to(DEVICE), yb.to(DEVICE).float()

        optimizer.zero_grad()
        loss = loss_fn(model(Xb), yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

# ==========================================================
# LOAD DATA
# ==========================================================
train_files = sorted([f for f in os.listdir(DATA_DIR) if "train" in f])

client_train = []
client_test = []
client_labels = []

for f in train_files:
    Xtr, ytr = load_csv(os.path.join(DATA_DIR, f))
    Xte, yte = load_csv(os.path.join(DATA_DIR, f.replace("train", "test")))

    client_train.append(make_loader(Xtr, ytr))
    client_test.append(make_loader(Xte, yte))
    client_labels.append(yte)

# global test
Xg, yg = load_csv(os.path.join(DATA_DIR, "test.csv"))
global_loader = make_loader(Xg, yg)

# ==========================================================
# INIT MODEL
# ==========================================================
config = BranchNetConfig(Xg.shape[1])
global_model = BranchNet(config).to(DEVICE)

print("Model initialized")

# ==========================================================
# FEDERATED LEARNING
# ==========================================================
for rnd in range(FL_ROUNDS):
    print(f"\nRound {rnd+1}")

    client_models = []
    local_accs = []

    for i in range(len(client_train)):
        m = BranchNet(config).to(DEVICE)
        m.load_state_dict(global_model.state_dict())

        loss = train_client(m, client_train[i])
        client_models.append(m)

        preds = eval_model(m, client_test[i])
        acc_val = accuracy(preds, client_labels[i])
        local_accs.append(acc_val)

        print(f"Client {i} | Loss {loss:.4f} | Local Acc {acc_val:.2f}%")

    print(f"Avg Local Accuracy: {np.mean(local_accs):.2f}%")

    # SAFE FEDAVG
    new_state = {}

    for key in global_model.state_dict():
        if global_model.state_dict()[key].dtype.is_floating_point:
            new_state[key] = sum(m.state_dict()[key] for m in client_models) / len(client_models)
        else:
            new_state[key] = global_model.state_dict()[key].clone()

    global_model.load_state_dict(new_state)

    preds = eval_model(global_model, global_loader)
    print(f"Global Accuracy: {accuracy(preds, yg):.2f}%")

print("Training complete")


Model initialized

Round 1
Client 0 | Loss 0.4769 | Local Acc 79.09%
Client 1 | Loss 0.3295 | Local Acc 74.02%
Client 2 | Loss 0.3363 | Local Acc 64.62%
Client 3 | Loss 0.6371 | Local Acc 51.55%
Client 4 | Loss 0.4820 | Local Acc 58.66%
Client 5 | Loss 0.6472 | Local Acc 60.25%
Avg Local Accuracy: 64.70%
Global Accuracy: 70.40%

Round 2
Client 0 | Loss 0.4565 | Local Acc 77.38%
Client 1 | Loss 0.1914 | Local Acc 73.98%
Client 2 | Loss 0.2472 | Local Acc 64.81%
Client 3 | Loss 0.6192 | Local Acc 51.34%
Client 4 | Loss 0.4410 | Local Acc 57.93%
Client 5 | Loss 0.6344 | Local Acc 58.78%
Avg Local Accuracy: 64.03%
Global Accuracy: 69.90%

Round 3
Client 0 | Loss 0.4484 | Local Acc 76.73%
Client 1 | Loss 0.1450 | Local Acc 73.27%
Client 2 | Loss 0.2122 | Local Acc 63.47%
Client 3 | Loss 0.6076 | Local Acc 51.12%
Client 4 | Loss 0.4227 | Local Acc 58.21%
Client 5 | Loss 0.6239 | Local Acc 59.87%
Avg Local Accuracy: 63.78%
Global Accuracy: 71.04%

Round 4
Client 0 | Loss 0.4457 | Local Acc 75